In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import os

def find_file(filename, search_path):
    for root, dirs, files in os.walk(search_path):
        if filename in files:
            return os.path.join(root, filename)
    return None

# 1. Automatically locate the file anywhere in your project
filename = "nigeria.csv"
# We search starting from two levels up to be safe
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
file_path = find_file(filename, project_root)

if file_path:
    print(f"Found it! Loading from: {file_path}")
    df = pd.read_csv(file_path)
else:
    # Last resort: list everything so we can see where it is
    print("Files in current folder:", os.listdir("."))
    raise FileNotFoundError(f"I searched everywhere but couldn't find {filename}. Is it spelled exactly like that?")

# 2. Add Country & Clean NASA Sentinel Values
df['Country'] = 'Nigeria'
df.replace(-999, np.nan, inplace=True)

# 3. Date Parsing
# NASA uses YEAR and Day of Year (DOY). 
df['Date'] = pd.to_datetime(df['YEAR'] * 1000 + df['DOY'], format='%Y%j')
df['Month'] = df['Date'].dt.month

# 4. Outlier Detection (Z-score > 3)
cols_to_check = ['T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR']
df_numeric = df[cols_to_check].dropna()
z_scores = np.abs(stats.zscore(df_numeric))

print("\n--- Success!Nigeria Data Profiled ---")
print(f"Total rows: {len(df)}")
print(f"Outlier Counts (|Z| > 3):\n{(z_scores > 3).sum()}")

# 5. Export Clean Data to a safe spot
df.to_csv("nigeria_clean.csv", index=False)
df.head()

Found it! Loading from: c:\Users\HP\Desktop\10acadamey\week0\-climate-challenge-week0\.github\workflows\nigeria.csv

--- Success!Nigeria Data Profiled ---
Total rows: 4108
Outlier Counts (|Z| > 3):
154


,YEAR,DOY,T2M,T2M_MAX,T2M_MIN,T2M_RANGE,PRECTOTCORR,RH2M,WS2M,WS2M_MAX,PS,QV2M,Country,Date,Month
0,2015,1,25.23,29.25,22.06,7.19,0.0,68.26,1.73,2.61,100.86,13.36,Nigeria,2015-01-01,1
1,2015,2,26.16,29.41,22.87,6.54,0.0,73.23,1.42,1.95,100.94,15.37,Nigeria,2015-01-02,1
2,2015,3,25.66,29.02,22.63,6.39,0.0,78.71,1.69,2.33,101.06,15.98,Nigeria,2015-01-03,1
3,2015,4,24.11,27.27,19.92,7.35,0.0,63.66,2.15,3.80,101.09,11.65,Nigeria,2015-01-04,1
4,2015,5,23.40,27.28,18.18,9.10,0.0,59.45,1.88,3.48,101.03,10.40,Nigeria,2015-01-05,1
